# 2편 편의점은 언제 바쁜가
**쉽게 따라 해 보면서 배우는 파이썬**

위에서부터 한 셀씩 `Shift + Enter`. 설명은 사이트의 2편 페이지에 있다.

## 2-1 질문을 먼저 적고 답을 한 줄로

In [ ]:
# 2편 실습 파일 묶음(zip)을 올리고 푼다
from google.colab import files
import zipfile, os
up = files.upload()
zipfile.ZipFile(list(up)[0]).extractall()
print(sorted(f for f in os.listdir() if f.startswith("cvs_")))

In [ ]:
import pandas as pd
enc = "utf-8-sig"
cvs = pd.read_csv("cvs_sales.csv", encoding=enc, parse_dates=["time"])
print(cvs.shape)
print(cvs.head(3))

In [ ]:
# 질문: 3월에 결제가 가장 많은 요일은? 가장 적은 요일과 몇 배 차이인가?
cvs["weekday"] = cvs["time"].dt.day_name().str[:3]
cnt = cvs["weekday"].value_counts()
print(cnt)
print(f"답: {cnt.idxmax()} 이 {cnt.max()}건으로 가장 많고, {cnt.idxmin()} 의 {cnt.max() / cnt.min():.2f}배다.")

## 2-2 한글 라벨이 깨지면 영어로 다시

In [ ]:
import matplotlib.pyplot as plt
by_cat = cvs.groupby("category")["amount"].sum() / 1e4      # 만원
by_cat.plot.bar(title="상품군별 매출", ylabel="만원", figsize=(5, 3))
plt.show()

In [ ]:
eng = {"과자": "Snack", "음료": "Drink", "도시락": "Lunchbox", "라면": "Ramen",
       "아이스크림": "Ice cream", "생활용품": "Daily goods"}
cvs["cat_en"] = cvs["category"].map(eng)
print("unmapped:", cvs["cat_en"].isna().sum())
by_cat = cvs.groupby("cat_en")["amount"].sum() / 1e4
by_cat.sort_values().plot.barh(title="Sales by category (March)", xlabel="KRW 10k", figsize=(5, 3))
plt.show()

## 2-3 평균이냐 합계냐 · CH-03

In [ ]:
t = cvs.groupby("cat_en")["amount"].agg(["mean", "sum", "count"])
t["sum"] = t["sum"] / 1e4
print(t.round(0).sort_values("sum", ascending=False))

In [ ]:
import seaborn as sns
order = t.sort_values("sum", ascending=False).index
fig, ax = plt.subplots(1, 2, figsize=(9, 3.2))
sns.barplot(data=cvs, x="cat_en", y="amount", order=order, errorbar=None, color="#028090", ax=ax[0])
ax[0].set(title="Default = mean per payment", xlabel="", ylabel="KRW")
cvs["amount_10k"] = cvs["amount"] / 1e4               # 합계는 커서 만원 단위로
sns.barplot(data=cvs, x="cat_en", y="amount_10k", order=order, estimator="sum", errorbar=None, color="#D98E22", ax=ax[1])
ax[1].set(title='estimator="sum" (March total)', xlabel="", ylabel="KRW 10k")
for a in ax: a.tick_params(axis="x", rotation=30)
fig.suptitle("CH-03 Sales by category: mean vs sum")
plt.tight_layout()
plt.show()

## 2-4 유통기한까지 며칠 남았나 · CH-04

In [ ]:
stock = pd.read_csv("cvs_stock.csv", encoding=enc, parse_dates=["expiry"])
today = pd.Timestamp("2025-03-31")
stock["days_left"] = (stock["expiry"] - today).dt.days
bins   = [-999, -1, 2, 7, 30, 9999]
labels = ["Expired", "0-2 days", "3-7 days", "8-30 days", "30+ days"]
stock["bucket"] = pd.cut(stock["days_left"], bins=bins, labels=labels)
bq = stock.groupby("bucket", observed=False)["qty"].sum()
print(bq)
print(stock[stock["days_left"] < 0][["product", "qty", "expiry"]])

In [ ]:
colors = ["#C9382A", "#E8A33A", "#F2C230", "#7FB8A4", "#028090"]
ax = bq.plot.bar(color=colors, figsize=(6, 3), rot=0,
                 title="CH-04 Items by days to expiry (Mar 31 night)", xlabel="", ylabel="Items")
ax.bar_label(ax.containers[0])
plt.show()

## 2-5 cumsum 으로 시재 다시 계산 · CH-05

In [ ]:
cash = pd.read_csv("cvs_cash.csv", encoding=enc, parse_dates=["date"])
opening = 200_000                                    # 3월 1일 아침 계산대 현금
cash["drawer"] = opening + (cash["cash_in"] - cash["cash_out"]).cumsum()
print(cash.head(4))
low = cash.loc[cash["drawer"].idxmin()]
print("lowest:", low["date"].date(), int(low["drawer"]))

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3))
ax.plot(cash["date"], cash["drawer"] / 1e4, color="#028090", marker=".")
ax.annotate(f"low {low['drawer'] / 1e4:.1f}", xy=(low["date"], low["drawer"] / 1e4),
            xytext=(low["date"] + pd.Timedelta(days=3), 30), arrowprops=dict(arrowstyle="->"))
ax.set(title="CH-05 CVS cash in drawer (end of day)", ylabel="KRW 10k", ylim=(0, 160))
import matplotlib.dates as mdates
ax.xaxis.set_major_formatter(mdates.DateFormatter("%m-%d"))   # 날짜는 월-일만
plt.show()

## 2-6 평균과 중앙값 · CH-06

In [ ]:
print("mean:  ", round(cvs["amount"].mean()))
print("median:", cvs["amount"].median())
print("top 5:", cvs["amount"].nlargest(5).tolist())

In [ ]:
k = cvs["amount"] / 1000
ax = sns.histplot(k, binwidth=1, binrange=(0, 30), color="#028090")
ax.axvline(k.mean(), color="#C9382A", linestyle="--", label=f"mean {k.mean():.1f}")
ax.axvline(k.median(), color="#1C2620", label=f"median {k.median():.1f}")
ax.set(title="CH-06 CVS payment amount (March)", xlabel="KRW thousand", ylabel="Payments")
ax.legend()
plt.show()

## 2-7 시간대 × 요일 피벗 · CH-07

In [ ]:
cvs["hour"] = cvs["time"].dt.hour
cvs["band"] = pd.cut(cvs["hour"], bins=[-1, 5, 9, 13, 17, 21, 23],
                     labels=["0-5", "6-9", "10-13", "14-17", "18-21", "22-23"])
days = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]
pv = cvs.pivot_table(index="band", columns="weekday", values="receipt", aggfunc="count", observed=False)
pv = (pv[days] / cvs.groupby("weekday")["time"].apply(lambda s: s.dt.date.nunique())[days]).round(0)   # 하루 평균
print(pv)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.6))
sns.heatmap(pv, annot=True, fmt=".0f", cmap="YlGnBu", ax=ax)
ax.set(title="CH-07 Payments per day: time band x weekday", xlabel="", ylabel="Hour")
plt.show()

## 2-8 영수증 금액의 첫 자리 · CH-08

In [ ]:
import numpy as np
first = cvs["amount"].astype(str).str[0].astype(int)
obs = first.value_counts(normalize=True).sort_index() * 100
d = np.arange(1, 10)
benford = pd.Series(np.log10(1 + 1 / d) * 100, index=d)
cmp_ = pd.DataFrame({"observed%": obs.round(1), "benford%": benford.round(1)})
print(cmp_)

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3))
ax.bar(d, obs, color="#028090", label="Observed")
ax.plot(d, benford, color="#D98E22", marker="o", label="Benford")
ax.set(title="CH-08 First digit of CVS payments", xlabel="First digit", ylabel="%", xticks=d)
ax.legend()
plt.show()

## 2-9 주말과 딱 떨어지는 금액 · CH-09

In [ ]:
nday = cvs.groupby("weekday")["time"].apply(lambda s: s.dt.date.nunique())
per_day = (cnt / nday)[days].round(0)
print(per_day)
cvs["round"] = cvs["amount"] % 10000 == 0           # 만원 단위로 딱 떨어지나
r = cvs[cvs["round"]]
print("round payments:", len(r))
print(r.groupby(["weekday", "pay"]).size().sort_values(ascending=False).head(4))
print("Sun 22-23h share:", round(((r["weekday"] == "Sun") & (r["hour"] >= 22)).mean() * 100), "%")

In [ ]:
ax = sns.countplot(data=r, x="weekday", order=days, hue="pay", palette=["#C9382A", "#028090", "#D98E22"])
ax.set(title="CH-09 Round-amount payments (x 10,000 KRW) by weekday", xlabel="", ylabel="Payments")
plt.show()